# N-real · 真实 CoT (TinyLlama): 直接答 vs 一步步想

> **小而真** (配套 reasoning-eval) · Chain-of-Thought 的核心主张: 让模型「一步步想」能提升推理。
> 这里用**真实 TinyLlama-1.1B-Chat** 在几道算术/逻辑题上对比「直接答」vs「CoT」, 并看到真实小模型的**真实出错**。
> CPU 离线确定性 (贪心), 生成很短。

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / "_shared"))
import realmodels as rm
import numpy as np, torch
print("真实模型可用性:", rm.available())

C:\Users\ericp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


真实模型可用性: {'gpt2': True, 'TinyLlama/TinyLlama-1.1B-Chat-v1.0': True}


> 若上面显示某模型为 False, 表示本机无该 HF 缓存, 真实例子会自动跳过 (不影响本专题的 toy notebook)。

## 1. 直接答 vs CoT (同一题, 两种提示)

In [2]:
tok, model = rm.tinyllama()
problems = [
    "A shop sells pens at 3 dollars each. I buy 7 pens and pay with a 50 dollar bill. How much change do I get?",
    "Tom is twice as old as Sara. Sara is 6. How old is Tom?",
    "If a train travels 60 km in 1.5 hours, what is its average speed in km/h?",
]
if model is not None:
    for q in problems:
        direct = rm.chat(tok, model, q + " Answer with just the final number.", max_new_tokens=24)
        cot    = rm.chat(tok, model, q + " Let's think step by step, then give the final number.", max_new_tokens=140)
        print("="*70)
        print("Q:", q)
        print("[直接答] ", direct.replace(chr(10),' ')[:120])
        print("[CoT  ]  ", cot.replace(chr(10),' ')[:400])
else:
    print("无 TinyLlama, 跳过")

Q: A shop sells pens at 3 dollars each. I buy 7 pens and pay with a 50 dollar bill. How much change do I get?
[直接答]  You get 25 cents change.
[CoT  ]   To calculate the amount of change you get, you need to subtract the 50 dollar bill from the total amount of money you paid. In this case, the total amount of money you paid is 50 dollars, so the amount of change you get is:  - 50 dollars - 3 dollars = 47 dollars  So, the amount of change you get is 47 dollars.


Q: Tom is twice as old as Sara. Sara is 6. How old is Tom?
[直接答]  Tom is twice as old as Sara, so the final number is 12.
[CoT  ]   To calculate Tom's age, we need to know Sara's age.  Sara is 6 years old.  Now, let's calculate Tom's age.  1. Sara is twice as old as Tom. 2. Tom is 6 years old. 3. Tom is twice as old as Sara.  So, Tom is 12 years old.  So, Tom's age is 12.


Q: If a train travels 60 km in 1.5 hours, what is its average speed in km/h?
[直接答]  The average speed of a train traveling 60 km in 1.5 hours is 35 km/
[CoT  ]   To find the average speed of a train traveling 60 km in 1.5 hours, we need to know its speed in km/h.  The speed of a train in km/h is calculated by multiplying its velocity (speed) by 1000.  In this case, the train's velocity is 100 km/h, so its speed in km/h is:  Speed in km/h = 100 km/h x 1000 = 1000 km/h  So, the average speed of the train in km/h is:  Average speed


> 观察: CoT 让模型**显式写出中间步骤**。对小模型, CoT 常能纠正直接答的错误 (把推理摊开,
> 每步更简单)。但**小模型仍会算错** —— 这正是你评估推理时要面对的真实情况。

## 2. 为什么 CoT 帮忙 (机制直觉) + 评估的坑

In [3]:
print('''CoT 为什么常有效 (机制直觉):
  - 直接答: 要求模型在「一步」内得出答案, 难题超出单步能力。
  - CoT:   把难题拆成多个简单步, 每步是模型更擅长的「下一词预测」, 错误率累积更低。
  - 代价:  更多 token = 更慢更贵; 且 CoT 可能「说一套算一套」(不忠实, 接 M12 CoT 忠实性)。

评估推理的真实坑 (你在 reasoning-eval 学的):
  - 只看最终答案对错, 会漏掉「蒙对」(过程错但答案对)。
  - 要评过程 → 需要 process reward / 步骤级标注 (process-reward 专题)。
  - 小模型 CoT 经常出现「步骤看着对、算术却错」—— 抓这种错正是评估的价值。''')

CoT 为什么常有效 (机制直觉):
  - 直接答: 要求模型在「一步」内得出答案, 难题超出单步能力。
  - CoT:   把难题拆成多个简单步, 每步是模型更擅长的「下一词预测」, 错误率累积更低。
  - 代价:  更多 token = 更慢更贵; 且 CoT 可能「说一套算一套」(不忠实, 接 M12 CoT 忠实性)。

评估推理的真实坑 (你在 reasoning-eval 学的):
  - 只看最终答案对错, 会漏掉「蒙对」(过程错但答案对)。
  - 要评过程 → 需要 process reward / 步骤级标注 (process-reward 专题)。
  - 小模型 CoT 经常出现「步骤看着对、算术却错」—— 抓这种错正是评估的价值。


## 3. 反思
- 你在**真实模型**上看到了 CoT 的效果与局限: 摊开推理常提分, 但小模型仍会错, 且 CoT 可能不忠实。
- 评估推理 ≠ 只看答案对错; 要看过程 (process reward) 和忠实性 (M12)。
- 这就是为什么「推理评估」是个独立难题: 生成能力上去了, **怎么可信地评它**反而成了瓶颈。